# HTMX Integration Example

This notebook demonstrates using cjm-tqdm-capture with HTMX for server-side rendering and SSE.

In [1]:
from fasthtml.common import *
import uuid, time

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.table import table, table_sizes
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_color
from cjm_fasthtml_tailwind.utilities.sizing import w, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from cjm_fasthtml_daisyui.core.testing import create_test_app

app, rt = create_test_app(theme=DaisyUITheme.CUPCAKE)

# Initialize
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Sample tasks
def file_processing_task():
    from tqdm import tqdm
    import time
    
    files = [f"file_{i}.txt" for i in range(50)]
    processed = []
    
    for file in tqdm(files, desc="Processing files"):
        time.sleep(0.05)  # Simulate file processing
        processed.append(file)
    
    return processed

def data_analysis_task():
    from tqdm import tqdm
    import time
    
    # Load data
    for _ in tqdm(range(30), desc="Loading data"):
        time.sleep(0.02)
    
    # Analyze
    for _ in tqdm(range(50), desc="Analyzing"):
        time.sleep(0.03)
    
    # Generate report
    for _ in tqdm(range(20), desc="Generating report"):
        time.sleep(0.02)
    
    return "Analysis complete"

In [5]:
# HTMX-powered UI - simplified without complex JavaScript
from cjm_fasthtml_daisyui.core.testing import create_test_page

def render_task_buttons(disabled=False):
    """Render task buttons with appropriate state"""
    btn_classes_files = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else "",
        m.r(2)
    )
    btn_classes_analysis = combine_classes(
        btn,
        btn_colors.secondary,
        btn_behaviors.disabled if disabled else ""
    )
    
    return Div(
        Button(
            "Process Files",
            hx_post="/start_task" if not disabled else None,
            hx_vals='{"task_type": "files"}' if not disabled else None,
            hx_target="#progress-container" if not disabled else None,
            hx_swap="innerHTML" if not disabled else None,
            cls=btn_classes_files,
            disabled=disabled,
            id="process-files-btn"
        ),
        Button(
            "Run Analysis",
            hx_post="/start_task" if not disabled else None,
            hx_vals='{"task_type": "analysis"}' if not disabled else None,
            hx_target="#progress-container" if not disabled else None,
            hx_swap="innerHTML" if not disabled else None,
            cls=btn_classes_analysis,
            disabled=disabled,
            id="run-analysis-btn"
        ),
        id="task-buttons",
        hx_swap_oob="true",  # Enable out-of-band swap
        cls=str(m.b(4))
    )

@rt("/")
def index():
    return create_test_page(
        "HTMX Progress Demo",
        Div(
            H1("HTMX + Polling Progress Tracking", cls=combine_classes(font_size._2xl, font_weight.bold, m.b(6))),
            
            # Task selection
            Div(
                H2("Select Task", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
                render_task_buttons(disabled=False),  # Start with enabled buttons
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Progress container
            Div(id="progress-container", cls=str(m.b(6))),
            
            # Active jobs
            Div(
                H2("Active Jobs", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
                Div(
                    id="jobs-container",
                    hx_get="/jobs_list",
                    hx_trigger="load, every 2s",
                    cls=str(overflow.x.auto)
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [6]:
# HTMX endpoints using FastHTML conventions
@rt("/start_task")
async def start_task(request):
    form = await request.form()
    task_type = form.get('task_type', 'files')
    
    job_id = str(uuid.uuid4())
    
    # Select task based on type
    if task_type == 'analysis':
        task = data_analysis_task
        task_name = "Data Analysis"
    else:
        task = file_processing_task
        task_name = "File Processing"
    
    # Start the job
    runner.start(
        job_id,
        task,
        patch_kwargs={
            "min_delta_pct": 2,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Return progress UI with disabled buttons and conditional polling
    return Div(
        # Disable buttons via out-of-band swap
        render_task_buttons(disabled=True),
        # Progress display
        Div(
            H3(f"{task_name} Progress", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
            
            # Status
            Div(
                f"Job ID: {job_id[:8]}...",
                id=f"job-status-{job_id}",
                cls=combine_classes(alert, alert_colors.info, m.b(4))
            ),
            
            # Progress content with conditional polling
            Div(
                # Initial empty state
                P("Overall: 0%", cls=str(font_weight.bold)),
                Progress(
                    value="0",
                    max="100",
                    cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
                ),
                # Initial load trigger only - polling added conditionally
                hx_get=f"/job_progress?job_id={job_id}",
                hx_trigger="load",
                hx_swap="innerHTML",
                id=f"progress-content-{job_id}"
            ),
            
            cls=combine_classes(card, bg_dui.base_200, p(6))
        )
    )

@rt("/job_progress")
def job_progress(job_id: str):
    """Returns the current progress state for a job"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        # Job not found or not started yet - continue polling
        return Div(
            P("Waiting for job to start...", cls=str(font_weight.bold)),
            Progress(value="0", max="100", cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))),
            # Keep polling until job starts
            hx_get=f"/job_progress?job_id={job_id}",
            hx_trigger="load delay:100ms",
            hx_swap="outerHTML"
        )
    
    # Build progress bars for each tqdm bar
    bars_html = []
    for bar_id, bar in snapshot['bars'].items():
        bars_html.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                  cls=str(font_size.sm)),
                Progress(
                    value=str(bar.progress), 
                    max="100", 
                    cls=combine_classes(progress, progress_colors.accent, w.full)
                ),
                cls=str(m.b(2))
            )
        )
    
    # Check if completed
    if snapshot['completed']:
        # Check if there are any other running jobs
        all_jobs = monitor.all()
        has_running = any(not job['completed'] for job in all_jobs.values() if job)
        
        # Return final state with conditionally re-enabled buttons and NO polling
        return Div(
            # Re-enable buttons if no other jobs are running
            render_task_buttons(disabled=has_running),
            # Final progress display
            P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
            Progress(
                value=str(snapshot['overall_progress']), 
                max="100", 
                cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
            ),
            *bars_html,
            Script(f"""
                // Update status to completed
                const statusEl = document.getElementById('job-status-{job_id}');
                if (statusEl) {{
                    statusEl.textContent = 'Task completed!';
                    statusEl.className = '{combine_classes(alert, alert_colors.success)}';
                }}
                // Refresh jobs list
                setTimeout(() => htmx.trigger('#jobs-container', 'refresh'), 1000);
            """)
            # No hx_trigger here - polling stops
        )
    
    # Task is in progress - continue polling
    return Div(
        P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
        Progress(
            value=str(snapshot['overall_progress']), 
            max="100", 
            cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
        ),
        *bars_html,
        # Continue polling while in progress
        hx_get=f"/job_progress?job_id={job_id}",
        hx_trigger="load delay:100ms",
        hx_swap="outerHTML"
    )

In [7]:
@rt("/jobs_list")
def jobs_list():
    jobs = monitor.all()
    
    if not jobs:
        return P("No active jobs", cls=str(text_color.gray(500)))
    
    return Table(
        Thead(
            Tr(
                Th("Job ID"),
                Th("Progress"),
                Th("Status"),
                Th("Bars")
            )
        ),
        Tbody(
            *[
                Tr(
                    Td(job_id[:8] + "..."),
                    Td(f"{job['overall_progress']:.1f}%"),
                    Td(
                        Span(
                            "Complete" if job['completed'] else "Running",
                            cls=combine_classes(
                                badge, 
                                badge_colors.success if job['completed'] else badge_colors.warning
                            )
                        )
                    ),
                    Td(str(len(job['bars'])))
                )
                for job_id, job in jobs.items()
            ]
        ),
        cls=combine_classes(table, table_sizes.xs, w.full)
    )

In [8]:
# Optional: SSE endpoint for direct streaming (not used in HTMX polling approach)
@rt("/stream")
def stream(job_id: str):
    """SSE endpoint - available for direct EventSource usage if needed"""
    return EventStream(
        sse_stream(
            monitor,
            job_id,
            interval=0.05,
            heartbeat=15.0,
            wait_for_start=True,
            start_timeout=10.0
        )
    )

In [9]:
# Alternative polling endpoint for HTMX (instead of SSE)
@rt("/poll")
def poll(job_id: str):
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        return Div("Job not found", cls=combine_classes(alert, alert_colors.error))
    
    # Return updated progress component
    bars_html = []
    for bar_id, bar in snapshot['bars'].items():
        bars_html.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                  cls=str(font_size.sm)),
                Progress(
                    value=str(bar.progress), 
                    max="100", 
                    cls=combine_classes(progress, progress_colors.accent, w.full)
                ),
                cls=str(m.b(2))
            )
        )
    
    # Include polling trigger only if not complete
    poll_trigger = Div(
        hx_get=f"/poll?job_id={job_id}",
        hx_trigger="load delay:500ms",
        hx_swap="outerHTML"
    ) if not snapshot['completed'] else None
    
    return Div(
        P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
        Progress(
            value=str(snapshot['overall_progress']), 
            max="100", 
            cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
        ),
        *bars_html,
        Div("Complete!", cls=combine_classes(alert, alert_colors.success)) if snapshot['completed'] else None,
        poll_trigger,
        id=f"poll-{job_id}"
    )

In [10]:
# Start server for Jupyter
from cjm_fasthtml_daisyui.core.testing import start_test_server
from fasthtml.jupyter import HTMX
from IPython.display import display

server = start_test_server(app)
display(HTMX())

In [11]:
# Stop server when done
server.stop()